# Hyperspectral Image Denoising using Wavelet Attention Network (MWAN)

This notebook demonstrates a complete deep learning pipeline for denoising Hyperspectral Images (HSI) using a custom PyTorch model (MWAN). It includes data loading, synthetic mixed noise generation, model training with automatic mixed precision (AMP), and evaluation with standard metrics (MPSNR, MSSIM, MSAM).


## 1. Imports and Setup
First, we import the necessary libraries. We'll use PyTorch for the deep learning components, and scikit-image for our evaluation metrics.


In [ ]:
import os
import copy
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

# Set device to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


## 2. Model Definition
Here we define the Multi-level Wavelet Attention Network (MWAN). It uses Discrete Wavelet Transforms (DWT) for downsampling and Inverse DWT for upsampling. The core of the network relies on a `WaveletAttentionBlock` that applies channel-wise attention to extract meaningful features from the noisy hyperspectral bands.


In [ ]:
class DWTForward(nn.Module):
    def __init__(self):
        super(DWTForward, self).__init__()
        kernel = torch.tensor([[[[1, 1], [1, 1]]], [[[-1, -1], [1, 1]]],
                               [[[-1, 1], [-1, 1]]], [[[1, -1], [-1, 1]]]], dtype=torch.float32) / 2.0
        self.register_buffer('kernel', kernel)
    def forward(self, x):
        b, c, h, w = x.shape
        kernel = self.kernel.repeat(c, 1, 1, 1)
        return F.conv2d(x, kernel, stride=2, groups=c)

class DWTInverse(nn.Module):
    def __init__(self):
        super(DWTInverse, self).__init__()
        kernel = torch.tensor([[[[1, 1], [1, 1]]], [[[-1, -1], [1, 1]]],
                               [[[-1, 1], [-1, 1]]], [[[1, -1], [-1, 1]]]], dtype=torch.float32) / 2.0
        self.register_buffer('kernel', kernel)
    def forward(self, x):
        b, c_total, h, w = x.shape
        c = c_total // 4
        kernel = self.kernel.repeat(c, 1, 1, 1)
        return F.conv_transpose2d(x, kernel, stride=2, groups=c)

class WaveletAttentionBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.dw_conv = nn.Conv2d(channels, channels, 3, padding=1, groups=channels)
        self.pw_conv = nn.Conv2d(channels, channels, 1)
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, channels // 8, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // 8, channels, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        res = x
        x = self.dw_conv(x)
        x = self.pw_conv(x)
        x = x * self.se(x)
        return x + res

class MWAN(nn.Module):
    def __init__(self, in_channels=8):
        super().__init__()
        self.dwt, self.idwt = DWTForward(), DWTInverse()
        self.head = nn.Conv2d(in_channels, 32, 3, padding=1)
        self.wave_process = nn.Sequential(
            WaveletAttentionBlock(32 * 4),
            nn.Conv2d(32 * 4, 32 * 4, 3, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            WaveletAttentionBlock(32 * 4)
        )
        self.tail = nn.Sequential(nn.Conv2d(32, 32, 3, padding=1), nn.LeakyReLU(0.2, inplace=True),
                                  nn.Conv2d(32, in_channels, 3, padding=1))

    def forward(self, x):
        identity = x
        b, c, h, w = x.shape
        pad_h, pad_w = h % 2, w % 2
        if pad_h > 0 or pad_w > 0: x = F.pad(x, (0, pad_w, 0, pad_h), mode='reflect')
        feat = self.head(x)
        wave_coeffs = self.dwt(feat)
        processed_coeffs = self.wave_process(wave_coeffs)
        reconstructed_feat = self.idwt(processed_coeffs)
        out = self.tail(reconstructed_feat)
        if pad_h > 0 or pad_w > 0: out = out[:, :, :h, :w]
        return out + identity



## 3. Noise Generation and Evaluation Metrics
To evaluate the robustness of our model, we simulate a realistic 'Mixed Noise' scenario on the hyperspectral images. This includes Gaussian noise, impulse noise, and structural noise like stripes and dead lines.
We also define standard metrics: Mean PSNR, Mean SSIM, and Mean Spectral Angle Mapper (MSAM).


In [ ]:
def add_mixed_noise(hsi, sigma=0.196, p_impulse=0.02, n_stripes=8, n_deadlines=4):
    H, W, B = hsi.shape
    noisy_hsi = hsi.copy()
    noisy_hsi += np.random.normal(0, sigma, hsi.shape)
    for i in range(B):
        mask = np.random.rand(H, W)
        noisy_hsi[mask < p_impulse/2, i] = 0
        noisy_hsi[mask > 1 - p_impulse/2, i] = 1
    stripe_cols = np.random.choice(range(W), n_stripes, replace=False)
    for col in stripe_cols:
        noisy_hsi[:, col, :] += np.random.uniform(-0.25, 0.25)
    deadline_cols = np.random.choice(range(W), n_deadlines, replace=False)
    for col in deadline_cols:
        noisy_hsi[:, col, :] = 0
    return np.clip(noisy_hsi, 0, 1)

def compute_all_metrics(origin, recon):
    bands = origin.shape[2]
    psnrs = [psnr(origin[:,:,i], recon[:,:,i], data_range=1.0) for i in range(bands)]
    ssims = [ssim(origin[:,:,i], recon[:,:,i], data_range=1.0) for i in range(bands)]
    org_f, rec_f = origin.reshape(-1, bands), recon.reshape(-1, bands)
    dot = np.sum(org_f * rec_f, axis=1)
    norm = np.linalg.norm(org_f, axis=1) * np.linalg.norm(rec_f, axis=1)
    msam = np.mean(np.degrees(np.arccos(np.clip(dot/(norm + 1e-8), -1, 1))))
    return np.mean(psnrs), np.mean(ssims), msam



## 4. Dataset and Subspace Projection
Because hyperspectral images are highly correlated across spectral bands, we project the data into a lower-dimensional subspace using Singular Value Decomposition (SVD) before passing it to the neural network. This speeds up training and improves denoising performance.
We also define a PyTorch `Dataset` to extract patches from the image for training.


In [ ]:
def subspace_projection_svd(noisy_hsi, clean_hsi, p=8):
    H, W, B = noisy_hsi.shape
    Y_flat = torch.from_numpy(noisy_hsi.reshape(H * W, B)).float().to(device)
    U, S, Vh = torch.linalg.svd(Y_flat, full_matrices=False)
    E = Vh[:p, :].T
    Z_noisy = torch.mm(U[:, :p], torch.diag(S[:p])).reshape(H, W, p).permute(2, 0, 1)
    X_flat = torch.from_numpy(clean_hsi.reshape(H * W, B)).float().to(device)
    Z_clean = torch.matmul(X_flat, E).reshape(H, W, p).permute(2, 0, 1)
    return Z_noisy.cpu(), Z_clean.cpu(), E.cpu()

class HSIPatchDataset(Dataset):
    def __init__(self, noisy_map, clean_map, patch_size=64, stride=32):
        self.patches_noisy, self.patches_clean = [], []
        _, h, w = noisy_map.shape
        for x in range(0, h - patch_size + 1, stride):
            for y in range(0, w - patch_size + 1, stride):
                self.patches_noisy.append(noisy_map[:, x:x+patch_size, y:y+patch_size])
                self.patches_clean.append(clean_map[:, x:x+patch_size, y:y+patch_size])
    def __len__(self): return len(self.patches_noisy)
    def __getitem__(self, idx): return self.patches_noisy[idx], self.patches_clean[idx]



## 5. Training and Evaluation Pipeline
This function coordinates the entire process:
1. Loading and normalizing the image.
2. Injecting mixed noise.
3. Extracting the SVD subspace and preparing the DataLoader.
4. Training the MWAN model using AdamW and a learning rate scheduler.
5. Performing inference and calculating the final metrics.
6. Plotting the results visually.


In [ ]:
def run_robustness_test(file_path):
    print(f"\n{'='*20} PROCESSING: {os.path.basename(file_path)} {'='*20}")

    if not os.path.exists(file_path):
        print(f"Error: {file_path} not found. Skipping..."); return

    # Load and Normalize
    orig = np.load(file_path).astype(np.float32)
    for i in range(orig.shape[2]):
        m, M = orig[:,:,i].min(), orig[:,:,i].max()
        if M > m: orig[:,:,i] = (orig[:,:,i] - m) / (M - m)

    # Mixed Noise Configuration
    sigma_val = 50.0 / 255.0
    p_impulse, n_stripes, n_deadlines = 0.02, 8, 4

    print("Generating Mixed Noise...")
    noisy = add_mixed_noise(orig, sigma=sigma_val, p_impulse=p_impulse,
                            n_stripes=n_stripes, n_deadlines=n_deadlines)

    # Subspace Training Setup
    p_val = 8
    Z_noisy, Z_clean, E = subspace_projection_svd(noisy, orig, p=p_val)
    z_mean, z_std = Z_noisy.mean(), Z_noisy.std()
    Z_noisy_n, Z_clean_n = (Z_noisy - z_mean)/z_std, (Z_clean - z_mean)/z_std

    dataloader = DataLoader(HSIPatchDataset(Z_noisy_n, Z_clean_n), batch_size=32, shuffle=True)
    model = MWAN(in_channels=p_val).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=15, factor=0.5)
    
    # We use GradScaler if CUDA is available for Automatic Mixed Precision
    scaler_amp = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None

    print(f"Training MWAN (Robustness Mode)...")
    for epoch in range(100): # Reduced from 1000 for demonstration purposes in notebook
        model.train()
        running_loss = 0.0
        for bn, bc in dataloader:
            bn, bc = bn.to(device), bc.to(device)
            optimizer.zero_grad()
            
            if scaler_amp:
                with torch.amp.autocast('cuda'):
                    pred = model(bn)
                    loss = F.l1_loss(pred, bc)
                scaler_amp.scale(loss).backward()
                scaler_amp.step(optimizer)
                scaler_amp.update()
            else:
                pred = model(bn)
                loss = F.l1_loss(pred, bc)
                loss.backward()
                optimizer.step()
                
            running_loss += loss.item()

        avg_loss = running_loss / len(dataloader)
        scheduler.step(avg_loss)

        # Show loss and learning rate
        if (epoch + 1) % 10 == 0:
            curr_lr = optimizer.param_groups[0]['lr']
            print(f"Epoch [{epoch+1:4d}/100] | Batch-Avg Loss: {avg_loss:.6f} | Current LR: {curr_lr:.2e}")

    # Inference & Metrics
    model.eval()
    with torch.no_grad():
        Z_hat = model(Z_noisy_n.to(device).unsqueeze(0)).squeeze(0).cpu()
    Z_hat = (Z_hat * z_std) + z_mean
    denoised_hsi = torch.matmul(Z_hat.permute(1, 2, 0).to(device), E.to(device).t()).cpu().numpy()

    n_p, n_s, n_sam = compute_all_metrics(orig, noisy)
    d_p, d_s, d_sam = compute_all_metrics(orig, denoised_hsi)

    print("\nRESULTS TABLE")
    print("-" * 55)
    print(f"{'Metric':<18} | {'Mixed Noisy':<15} | {'Denoised'}")
    print(f"{'MPSNR (dB)':<18} | {n_p:<15.2f} | {d_p:.2f}")
    print(f"{'MSSIM':<18} | {n_s:<15.4f} | {d_s:.4f}")
    print(f"{'MSAM (deg)':<18} | {n_sam:<15.2f} | {d_sam:.2f}")
    print("-" * 55)

    # Visualization
    b = min(50, orig.shape[2]-1)
    plt.figure(figsize=(15, 5))
    imgs = [orig[:,:,b], noisy[:,:,b], denoised_hsi[:,:,b]]
    titles = ["Original", "Mixed Noise Input", "MWAN Denoised"]
    for i, (img, t) in enumerate(zip(imgs, titles)):
        plt.subplot(1, 3, i+1); plt.imshow(img, cmap='gray'); plt.title(t); plt.axis('off')
    plt.tight_layout(); plt.show()



## 6. Run the Experiment
Finally, we define the dataset paths and execute our testing loop over the datasets.


In [ ]:
# Define dataset paths
dataset_paths = [
    '/content/dc_256x256x191.npy',
    '/content/indianpinearray.npy',
    '/content/PaviaU.npy',
    '/content/Pavia_resized_1.npy',
    '/content/KSC.npy'
]

# Run tests
for path in dataset_paths:
    try:
        run_robustness_test(path)
    except Exception as e:
        print(f"Failed to process {path}: {e}")

